In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
## supress warning
import warnings 
warnings.simplefilter('ignore')

In [ ]:
df = pd.read_csv('/kaggle/input/alzheimer-features/alzheimer.csv')
df.head()

# Missing Value Detection and imputation

In [ ]:
## Check for missing_values
import seaborn as sns
sns.heatmap(df.isna(),cmap = 'mako')
df.isna().sum()

In [ ]:
## Impute missing values
from sklearn.impute import KNNImputer
knn = KNNImputer()
df['SES'] = knn.fit_transform(df[['SES']])
df['MMSE'] = knn.fit_transform(df[['MMSE']])

# Outlier Detection

In [ ]:
## create a function to give index of outlier columns
def outlier_idx(df,col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    
    upper_bound = q3 + 1.5 * iqr 
    lower_bound = q1 - 1.5 * iqr
    
    ls = df.index[(df[col] > upper_bound) | (df[col]<lower_bound)]
    return ls

def imputer(df):
    num_cols = list(df.select_dtypes(include = ['int','float']).columns)
    outlier_list = []
    
    for i in num_cols:
        idx = outlier_idx(df,i)
        df.loc[idx,i] = np.nan
        
    knn = KNNImputer(n_neighbors= 5)
    df[num_cols] = knn.fit_transform(df[num_cols])
    return df


df_clean = df.copy()
df_clean = imputer(df)

# Univariate Data Analysis

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
cat_cols = list(df_clean.select_dtypes(include = ['O']).columns)
num_cols = list(df_clean.select_dtypes(include = ['int','float']).columns)

In [ ]:
## Univariate analysis for numerical columns
n_rows = 8
n_col = 3
fig, axes = plt.subplots(nrows = n_rows, ncols = n_col,figsize = (18,40))
for i,col in enumerate(num_cols):
    
    sns.histplot(data = df_clean, x = col, hue = 'Group', ax = axes[i,0],palette = 'Set2')
    axes[i,0].set_title(f'Histogram for {col} column')
    
    sns.boxplot(data = df_clean, x = col,hue = 'Group', ax = axes[i,1],palette = 'Set2')
    axes[i,1].set_title(f'Boxplot for {col} column')
    
    sns.kdeplot(data = df_clean, x = col,hue = 'Group', ax = axes[i,2],palette = 'Set2')
    axes[i,2].set_title(f'KdePlot for {col} column')

In [ ]:
## Univariate analysis of categorical columns
from collections import Counter

n_rows = len(cat_cols)
n_col = 2
fig, axes = plt.subplots(nrows = n_rows, ncols = n_col,figsize = (12,10))

for i,col in enumerate(cat_cols):
    
    sns.countplot(data = df_clean, x = col, hue = 'Group', ax = axes[i,0],palette = 'Set2')
    axes[i,0].set_title(f'Countplot for {col} column')
    
    # Count occurrences of each category
    count = Counter(df_clean[col])

    # Extract labels and sizes from the counter
    labels = list(count.keys())
    sizes = list(count.values())

    
    
    axes[i,1].pie(x = sizes, labels = labels, autopct='%1.1f%%', startangle=140, wedgeprops={'width': 0.4}, pctdistance = 0.8, colors = ['mediumseagreen','lightcoral','lightsteelblue'])
    axes[i,1].set_title(f'Piechart for {col} column')
    
    

#### ⭐ Summary of Univariate Analysis:
* Dementia is More Common Among men after the age of 68
* People with dementia scored between 10-20 on their MMSE exam
* The Average Age around which Dementia is more likely to occur is around 76
* People of higher social economic status are more likely to be demented

# Bivariate  and Multivariate Analysis

In [ ]:
ax = sns.barplot(data = df_clean, x=  'M/F',y = 'Age',hue = 'Group',palette = 'Set2')
plt.title('Barplot for Different groups by gender and Age')
plt.tight_layout()
for p in ax.patches:
    height = int(p.get_height())
    plt.annotate(f'{height}',
                 xy = (p.get_x() + p.get_width() / 2, height),
                 xytext = (0,9),
                 ha = 'center',
                 va = 'bottom',
                 textcoords='offset points',
               )

In [ ]:
ax = sns.barplot(data = df_clean, x = 'M/F', y = 'eTIV', hue = 'Group',palette = 'Set2')
for p in ax.patches:
    height = int(p.get_height())
    plt.annotate(f'{height}',
                 xy = (p.get_x() + p.get_width() / 2, height),
                 xytext = (0,-50),
                 ha = 'center',
                 va = 'bottom',
                 textcoords='offset points',
               )
plt.title('InterCranial Volume Among Different Groups By Gender')

In [ ]:
ax = sns.barplot(data = df_clean, x = 'M/F', y= 'EDUC', hue = 'Group', palette = 'Set2')
for p in ax.patches:
    height = int(p.get_height())
    plt.annotate(f'{height}',
                 xy = (p.get_x() + p.get_width() / 2, height),
                 xytext = (0,-50),
                 ha = 'center',
                 va = 'bottom',
                 textcoords='offset points',
               )
plt.title('Years Studied By Different Group')


In [ ]:
sns.pairplot(df_clean,hue = 'Group')
plt.title('Pairplot of different categories')

In [ ]:
## correlation between different numerical cols
sns.heatmap(df_clean[num_cols].corr(), cmap = 'mako', annot = True)

#### ⭐⭐ Summary of Multivariate and bivariate analysis
* men have higher inter cranial volume than female
* People with dementia have lower Inter-Cranial Volume Than Normal ones
* People Who studied more often had lower chances of dementia


# Data Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [ ]:
df_clean = pd.get_dummies(data = df_clean, columns = ['M/F'])

In [ ]:
df_clean

In [ ]:
X = df_clean.drop(columns = ['Group'])
y = le.fit_transform(df_clean['Group'])

# Data Splitting And Multiple Model Selection

In [ ]:
## Importing necessary classes and models
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## Performance Metrics
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, shuffle = True, random_state = 42)

In [ ]:
model_grid = {
    'RandomForest' : RandomForestClassifier(),
    'KNN' : KNeighborsClassifier(),
    'XGBoost' : XGBClassifier(),
    'Light GBM' : LGBMClassifier(verbose = -1)
}

In [ ]:
for model in model_grid:
    m = model_grid[model]
    m.fit(x_train, y_train)
    
    train_pred = m.predict(x_train)
    test_pred = m.predict(x_test)
    
    ## Training accuracy
    train_acc = accuracy_score(y_train, train_pred)
    train_f1 = f1_score(y_train, train_pred,  average = 'weighted')
    train_recall = recall_score(y_train, train_pred,   average = 'weighted')
    train_precision = precision_score(y_train , train_pred, average = 'weighted')
    
    ## Test accuracy
    test_acc = accuracy_score(y_test, test_pred)
    test_f1 = f1_score(y_test, test_pred,  average = 'weighted')
    test_recall = recall_score(y_test, test_pred,   average = 'weighted')
    test_precision = precision_score(y_test , test_pred, average = 'weighted')
    
    print('-' * 40)
    
    print(f'<---- Stats For {model} ---->')
    print('\n')
    
    print(f'Train Set Scoring ---------> ')
    print(f'-Accuracy Score : {train_acc}')
    print(f'-F1 score : {train_f1}')
    print(f'-Recall score ; {train_recall}')
    print(f'-Precision : {train_precision}')
    
    print('\n')
    
    print(f'Test Set Scoring ---------> ')
    print(f'-Accuracy Score : {test_acc}')
    print(f'-F1 score : {test_f1}')
    print(f'-Recall score ; {test_recall}')
    print(f'-Precision : {test_precision}')
    
    print('-' * 40)
    
    

# Model HyperParameter Tuning

In [ ]:
## Random Forest HyperParameters
rf_params = {
    'min_samples_leaf': [x for x in range(15)],
    'min_samples_split' : [x for x in range(20)],
    'n_estimators' : [x for x in range(100,1500,100)],
    'max_depth' : [1,2,3,4,5,6,7,8,9,10],
    'criterion' : ['gini','entropy','log_loss'],
    'max_features' : ['auto','sqrt','log2'],
    'bootstrap' : [True]
}

## XGBOOST hyperparameters

xg_params  = {
    'subsample': [0.5, 0.55,0.60,0.65,0.7,0.75,0.8,0.85,0.9,1],
    'colsample_bytree' : [0.5, 0.55,0.60,0.65,0.7,0.75,0.8,0.85,0.9,1],
    'colsample_bynode' : [0.5, 0.55,0.60,0.65,0.7,0.75,0.8,0.85,0.9,1],
    'n_estimators' : [x for x in range(100,1500,100)],
    'max_depth' : [1,2,3,4,5,6,7,8,9,10],
    'reg_alpha' : [x for x in range(1,101,1)],
    'reg_lambda' : [x for x in range(1,101,1)],
    'learning_rate' : [0.01,0.05,0.03,0.1,0.3,0.5,1] 
}

light_params = {
    'feature_fraction': [0.5, 0.55,0.60,0.65,0.7,0.75,0.8,0.85,0.9,0.95],
    'n_estimators' : [x for x in range(100,1500,100)],
    'max_depth' : [1,2,3,4,5,6,7,8,9,10],
    'lambda_l1' : [x for x in range(1,101,1)],
    'lambda_l2' : [x for x in range(1,101,1)],
    'boosting_type' : ['gbdt', 'rf'],
    'verbose' :[-1],
    'min_data_in_leaf': [x for x in range(15)],
}

In [ ]:
results = {}
model_params = [
    ('Random Forest',RandomForestClassifier(),rf_params),
    ('XGBoost', XGBClassifier(), xg_params),
    ('LightBoost', LGBMClassifier(),light_params)
]

for name, model, params in model_params:
    cv = RandomizedSearchCV(estimator = model, n_iter = 50, cv = 3,param_distributions = params)
    cv.fit(x_train, y_train)
    pred = cv.predict(x_test)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average = 'weighted')
    precision = precision_score(y_test, pred, average = 'weighted')
    recall = recall_score(y_test, pred, average = 'weighted')
    results[name] = {'accuracy':acc, 'precision':precision, 'recall': recall, 'f1 score' : f1, 'best param':cv.best_params_}
    
    

In [ ]:
results

# The Best accuracy That we achieved is around 0.8533 By lightboost

### Conclusion
* I will try to further increase the accuracy of the Models By trying out different approaches. 
* Please be sure to comment to advise me how to change my approach. 
* I will keep trying my best. Thanks !!!!!